# Generate Large Model Embeddings for ArXiv Papers

This notebook runs large embedding models on Google Colab's GPU that are too heavy for local machines.

**Available models:**
- `Qwen/Qwen3-Embedding-4B` (4B params)
- `Qwen/Qwen3-Embedding-8B` (8B params, #1 on MTEB multilingual)
- `intfloat/e5-mistral-7b-instruct` (7B params)

**Steps:**
1. Upload your `arxiv_papers.db` database
2. Select which model(s) to run
3. Generate embeddings using Colab's T4/A100 GPU
4. Download the embeddings

**Requirements:** Colab with GPU runtime (T4 minimum, A100 recommended for 8B)

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Install dependencies
!pip install -q sentence-transformers torch numpy tqdm

In [ ]:
# Upload your database file
from google.colab import files
print("Upload your arxiv_papers.db file:")
uploaded = files.upload()

In [ ]:
import sqlite3
import numpy as np
from sentence_transformers import SentenceTransformer
import json
import gc
import torch
from datetime import datetime

DB_PATH = 'arxiv_papers.db'

# ===== ALL 3 QWEN EMBEDDING MODELS =====
MODELS_TO_RUN = [
    {'name': 'Qwen/Qwen3-Embedding-0.6B', 'batch_size': 128},
    {'name': 'Qwen/Qwen3-Embedding-4B', 'batch_size': 16},
    {'name': 'Qwen/Qwen3-Embedding-8B', 'batch_size': 8},
]

print(f"Models to run: {[m['name'] for m in MODELS_TO_RUN]}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

In [ ]:
# Load papers from database
def load_papers(db_path):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("""
        SELECT id, title, abstract, primary_category
        FROM papers
        WHERE abstract IS NOT NULL AND abstract != ''
        ORDER BY id
    """)
    papers = []
    for row in cursor.fetchall():
        papers.append({
            'id': row[0],
            'title': row[1],
            'abstract': row[2],
            'primary_category': row[3]
        })
    conn.close()
    return papers

def combine_text(title, abstract):
    if abstract:
        return f"{title}. {abstract}"
    return title

papers = load_papers(DB_PATH)
texts = [combine_text(p['title'], p['abstract']) for p in papers]
paper_ids = [p['id'] for p in papers]

print(f"Loaded {len(papers)} papers")
print(f"Sample: {texts[0][:100]}...")

In [ ]:
def generate_and_save(model_name, batch_size, texts, paper_ids):
    """Generate embeddings for a single model and save results."""
    print(f"\n{'='*60}")
    print(f"Model: {model_name}")
    print(f"Batch size: {batch_size}")
    print(f"{'='*60}")

    safe_name = model_name.replace('/', '-')

    # Load model
    print(f"Loading model...")
    model = SentenceTransformer(model_name, trust_remote_code=True)
    model = model.to('cuda' if torch.cuda.is_available() else 'cpu')
    dim = model.get_sentence_embedding_dimension()
    print(f"Model loaded! Dimension: {dim}")

    # Generate embeddings
    print(f"Generating embeddings for {len(texts)} texts...")
    start_time = datetime.now()

    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    duration = (datetime.now() - start_time).total_seconds()
    print(f"Done! Shape: {embeddings.shape}, Time: {duration:.1f}s, Rate: {len(texts)/duration:.2f} papers/s")

    # Save embeddings
    np.save(f'{safe_name}_embeddings.npy', embeddings)

    # Save metadata
    metadata = {
        'model_name': model_name,
        'dimension': int(embeddings.shape[1]),
        'num_papers': len(papers),
        'generated_at': datetime.now().isoformat(),
        'paper_ids': paper_ids,
        'processing_time_seconds': duration,
        'papers_per_second': len(texts) / duration
    }
    with open(f'{safe_name}_metadata.json', 'w') as f:
        json.dump(metadata, f, indent=2)

    print(f"Saved: {safe_name}_embeddings.npy ({embeddings.nbytes / 1024 / 1024:.1f} MB)")
    print(f"Saved: {safe_name}_metadata.json")

    # Cleanup GPU memory
    del model, embeddings
    gc.collect()
    torch.cuda.empty_cache()
    print(f"GPU memory cleared.")

    return safe_name

In [ ]:
# Run all selected models
generated_files = []

for model_config in MODELS_TO_RUN:
    try:
        safe_name = generate_and_save(
            model_config['name'],
            model_config['batch_size'],
            texts,
            paper_ids
        )
        generated_files.append(safe_name)
    except Exception as e:
        print(f"ERROR with {model_config['name']}: {e}")

print(f"\n\nCompleted {len(generated_files)}/{len(MODELS_TO_RUN)} models!")
for name in generated_files:
    print(f"  - {name}_embeddings.npy")
    print(f"  - {name}_metadata.json")

In [ ]:
# Download all generated files
from google.colab import files

for name in generated_files:
    print(f"\nDownloading {name}...")
    files.download(f'{name}_embeddings.npy')
    files.download(f'{name}_metadata.json')

## After Download

Move the downloaded files to your local machine:

```bash
# For Qwen3-Embedding-4B
mkdir -p ArXiv/data/embeddings/Qwen-Qwen3-Embedding-4B/
mv Qwen-Qwen3-Embedding-4B_embeddings.npy ArXiv/data/embeddings/Qwen-Qwen3-Embedding-4B/embeddings.npy
mv Qwen-Qwen3-Embedding-4B_metadata.json ArXiv/data/embeddings/Qwen-Qwen3-Embedding-4B/metadata.json

# For Qwen3-Embedding-8B
mkdir -p ArXiv/data/embeddings/Qwen-Qwen3-Embedding-8B/
mv Qwen-Qwen3-Embedding-8B_embeddings.npy ArXiv/data/embeddings/Qwen-Qwen3-Embedding-8B/embeddings.npy
mv Qwen-Qwen3-Embedding-8B_metadata.json ArXiv/data/embeddings/Qwen-Qwen3-Embedding-8B/metadata.json
```

Then add the models to `config.py`:

```python
EMBEDDING_MODELS = {
    # ... existing models ...
    'Qwen/Qwen3-Embedding-4B': 1024,
    'Qwen/Qwen3-Embedding-8B': 1024,
}
```

Then re-run PCA clustering:

```bash
python scripts/04_pca_clustering.py --all-models
```